# Stg Layer — restaurant_geocode (locate the restaurants named in videos)

Look up each restaurant extracted from the videos and write the answer into the **geocode columns of `${silver}.stg_restaurant_mentions`** (`match_status`, `place_id`, `place_name`, `formatted_address`, `geo_ward`, `geo_city`, `latitude`, `longitude`, `place_types`, `name_similarity`, `geo_query_text`, `geocoded_at`). One lookup per restaurant key = `restaurant_name_normalized` + `district_norm`, copied to every mention row of that key; `stg/restaurant_mentions` keeps them on its full sync and copies them to new videos of a known key.

**Since 2026-11-06 (datle) there is no separate cache table:** the lookups of the retired `stg_restaurant_geocode` were copied into `stg_restaurant_mentions` once (by `stg/restaurant_mentions`). Rejected candidates (`not_food`, `name_mismatch`, `no_result`) keep only `match_status`, the other columns stay NULL.

**Flow:**
1. Distinct restaurant keys from `stg_restaurant_mentions` (representative name / address = latest mention with an address)
2. Keys **not looked up yet** (`match_status` IS NULL on every row of the key) → call the API, within the **caps** (since 2026-11-02: `MONTHLY_CALL_LIMIT` calls per Vietnam month and at most `DAILY_CALL_LIMIT` per day, counted over every run in `stg_restaurant_geocode_calls`); the rest wait for the next day
3. Pick the first candidate that is a food place (`restaurant`, `food`, `cafe`, `bakery`, `*_restaurant`, ...) **and** has a similar name (unaccented token overlap ≥ `MIN_NAME_SIMILARITY`); otherwise store NULLs with a `match_status` (`no_result`, `not_food`, `name_mismatch`)
4. MERGE into `stg_restaurant_mentions` (only rows not looked up yet, or `osm_approx` → `places_match`). HTTP errors are **not** stored, so they are retried next run.

**Quota:** on HTTP 429 (e.g. `SearchTextRequest per day` used up) the run stops calling the API but does **not** fail: what was found is cached, the rest waits for the next run, and the restaurant stays out of `dim_restaurant` meanwhile (its mentions count as Unknown). A rejected key (401 / 403) fails the task.

**Address fields:** Google returns the post-2025 wards (e.g. "Vĩnh Hội") and no district, so `geo_ward` / `geo_city` / `latitude` / `longitude` / `place_id` / `formatted_address` come from Google and the district stays the one stated in the video.

**Secret:** `maps/api_key` (Places API (New)). Uses `requests` (no psycopg, no FIPS issue).

**Cost:** the field mask only asks for Pro-tier fields (id, name, address, location, types, address components), so each call is billed as **Text Search Pro**: 5,000 free calls per month, then $32 per 1,000. Asking for `rating` / `userRatingCount` would make every call **Enterprise** (1,000 free, then $35 per 1,000). Text Search Pro's 5,000 free calls per month are shared with `auto_search` (mode places, Pro, its own cap 1,500 / month), so geocode gets **3,000 per month** (`MONTHLY_CALL_LIMIT`, 500 left as margin) and up to **300 per day** (`DAILY_CALL_LIMIT`), so a backlog clears in a day or two instead of waiting at 100 per day. The GCP quota `SearchTextRequest per day` must allow it (shared with auto_search).

**Since 2026-11-03 (datle): no Google calls by default, free lookups instead.** Google enforces 100 Text Search calls per day for the whole GCP project in Vietnam, and datle gave all of them to `auto_search` (one call = up to 20 places). So `DAILY_CALL_LIMIT = 0` and each new key goes through two free steps:
1. **Photon** (OpenStreetMap geocoder, `photon.komoot.io`, ~1 request / second, at most `PHOTON_MAX_PER_RUN` per run) on the address said in the video → approximate coordinates. Only keys with an address that has a number are looked up. Dev test on 60 addresses vs Google: 56 found, 17 within 200 m, 35 within 1 km, median 558 m — good for ward / district, approximate as a map pin.
2. **Match with a Google place already found by auto_search** (`stg_places_latest`, no call): name similarity ≥ `PLACES_MIN_SIMILARITY`, at least one **distinctive** word in common (not a dish / place word from `GENERIC_NAME_WORDS`, so "Bún riêu cua" never matches) and within `PLACES_MATCH_MAX_M` (500 m) of the Photon point; no address in the video = no match (chains have several branches) → `match_status = places_match` with the Google `place_id`, so the restaurant gets Google's rating / location and merges with the Google-only row.
3. Otherwise `match_status = osm_approx` (Photon point, no `place_id`): `gold/restaurants` keys it by name + district and the dashboard shows "vị trí gần đúng". Every run re-tries the `osm_approx` rows against new Google places (step 2) and upgrades them.


In [ ]:
%run ../gold/_udfs


In [ ]:
# --- Config ---
# --- Layer schemas: job parameters bronze / silver / gold (catalog.schema), default prod.* ---
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))

MENTIONS_TABLE = f"{SILVER}.stg_restaurant_mentions"
CALLS_TABLE = f"{SILVER}.stg_restaurant_geocode_calls"  # API calls per run, for the daily cap
DAILY_CALL_LIMIT = 0            # Google Places calls per Vietnam day (2026-11-03: 0, the 100/day GCP quota goes to auto_search)
MONTHLY_CALL_LIMIT = 3000       # per Vietnam month: 5,000 free Pro calls - 1,500 auto_search Pro cap - 500 margin
SNAPSHOT_TZ = "Asia/Ho_Chi_Minh"
MIN_NAME_SIMILARITY = 0.5
HCMC_BIAS = {"circle": {"center": {"latitude": 10.7769, "longitude": 106.7009}, "radius": 30000.0}}
# Pro-tier fields only: adding rating / userRatingCount would bill every call as Text Search Enterprise
# (1,000 free calls / month instead of 5,000).
FIELD_MASK = ("places.id,places.displayName,places.formattedAddress,places.location,places.types,"
              "places.addressComponents")
PHOTON_URL = "https://photon.komoot.io/api/"   # free OpenStreetMap geocoder (fair use: ~1 request / second)
PHOTON_MAX_PER_RUN = 300        # Photon lookups per run (the refresh job runs every 3 h, so a backlog clears in a day)
PHOTON_BUDGET_SECONDS = 15 * 60  # 2026-11-06: stop Photon lookups after this (a slow Photon made the task hit its 30 min timeout)
PHOTON_TIMEOUT_SECONDS = 10
PHOTON_HEADERS = {"User-Agent": "project1-restaurant-finder/1.0 (personal, low volume)"}
HCMC_BBOX = "106.35,10.35,107.05,11.20"     # Photon bbox (minLon,minLat,maxLon,maxLat) for Hồ Chí Minh addresses
PLACES_TABLE = f"{SILVER}.stg_places_latest"  # Google places found by auto_search / manual searches (stg/places)
PLACES_MIN_SIMILARITY = 0.8     # name similarity to take a Google place's place_id without a call
PLACES_MATCH_MAX_M = 500        # ... and at most this far from the Photon point (1,500 m gave wrong matches, 2026-11-03)
# Words that only name a dish / kind of place: a name made of these alone ("Bún riêu cua", "Bánh canh cua 20k") is not
# specific enough to take a Google place's id; the name must share at least one other word with the Google name
GENERIC_NAME_WORDS = set("""
quan nha hang tiem an uong quan_an bun bo hue rieu cua oc banh canh mi pho com tam ga vit heo thit nuong chien
ca kho lau hai san chay chao long hu tieu xao xeo cuon trang nem goi xoi che kem tra sua cafe ca phe coffee
cha gio de lon suon bi bep mon ngon dac san sai gon ha noi dimsum sushi pizza bbq buffet steak bit tet nuoc
mam cay rau vat beo dau phu dau hu vietnamese food restaurant kitchen
""".split())
FOOD_TYPES = {"restaurant", "food", "cafe", "coffee_shop", "bakery", "bar", "meal_takeaway", "meal_delivery",
              "food_court", "dessert_shop", "ice_cream_shop", "tea_house", "juice_shop", "snack_bar", "diner",
              "bistro", "deli", "pub"}

print(f"Mentions source: {MENTIONS_TABLE}")
print(f"Geocode columns written into {MENTIONS_TABLE} · max {DAILY_CALL_LIMIT} API calls per day, {MONTHLY_CALL_LIMIT} per month")

# Geocode columns of stg_restaurant_mentions (lookup result -> column)
GEO_COLS = {"match_status": "STRING", "place_id": "STRING", "place_name": "STRING", "formatted_address": "STRING",
            "geo_ward": "STRING", "geo_city": "STRING", "latitude": "DOUBLE", "longitude": "DOUBLE",
            "place_types": "ARRAY<STRING>", "name_similarity": "DOUBLE", "geo_query_text": "STRING"}
RENAME = {"ward": "geo_ward", "city": "geo_city", "query_text": "geo_query_text"}
# One row per restaurant key with its lookup (identical on every mention row of the key)
KEYS_SQL = f"""
    SELECT restaurant_name_normalized, district_norm,
           max_by(match_status, geocoded_at) AS match_status, max_by(geo_query_text, geocoded_at) AS query_text,
           max_by(place_name, geocoded_at) AS place_name, max_by(geo_city, geocoded_at) AS city,
           max_by(latitude, geocoded_at) AS latitude, max_by(longitude, geocoded_at) AS longitude
    FROM {MENTIONS_TABLE} WHERE match_status IS NOT NULL
    GROUP BY restaurant_name_normalized, district_norm
"""


In [ ]:
# --- Restaurant keys not looked up yet (district_norm is written by stg/restaurant_mentions, same rule as Gold) ---
todo = spark.sql(f"""
    WITH m AS (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY restaurant_name_normalized, district_norm
                               ORDER BY address IS NULL, extracted_at DESC, video_id) AS rn,
            MAX(match_status) OVER (PARTITION BY restaurant_name_normalized, district_norm) AS key_status
        FROM {MENTIONS_TABLE}
    )
    SELECT m.restaurant_name_normalized, m.district_norm, m.restaurant_name, m.address, m.district, m.city
    FROM m
    WHERE m.rn = 1 AND m.key_status IS NULL
    ORDER BY m.restaurant_name_normalized
""").collect()

# --- Daily cap: calls already made today (Vietnam day) by earlier runs ---
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CALLS_TABLE} (
        run_ts    TIMESTAMP,
        call_date DATE,
        api_calls INT,
        cached    INT,
        errors    INT
    )
    USING DELTA
    COMMENT 'Places API calls per geocode run; sums per day / month enforce DAILY_CALL_LIMIT / MONTHLY_CALL_LIMIT.'
""")
today = spark.sql(f"SELECT to_date(from_utc_timestamp(current_timestamp(), '{SNAPSHOT_TZ}')) AS d").collect()[0]["d"]
used_today = spark.sql(f"SELECT COALESCE(SUM(api_calls), 0) AS n FROM {CALLS_TABLE} WHERE call_date = DATE'{today}'").collect()[0]["n"]
used_month = spark.sql(f"""SELECT COALESCE(SUM(api_calls), 0) AS n FROM {CALLS_TABLE}
                           WHERE trunc(call_date, 'MM') = trunc(DATE'{today}', 'MM')""").collect()[0]["n"]
budget = max(0, min(DAILY_CALL_LIMIT - used_today, MONTHLY_CALL_LIMIT - used_month))
print(f"Restaurant keys not looked up yet: {len(todo)} · calls today: {used_today}/{DAILY_CALL_LIMIT} · "
      f"this month: {used_month}/{MONTHLY_CALL_LIMIT} · this run may call: {budget}")


In [ ]:
# --- Call Places API (New) Text Search and pick the first acceptable candidate ---
import difflib
import re

import requests

API_KEY = dbutils.secrets.get("maps", "api_key") if budget > 0 else None


def name_tokens(s):
    return set(re.sub(r"[^a-z0-9 ]", " ", (unaccent_vi(s or "")).lower()).split())


def name_similarity(ours, theirs):
    """Max of the character ratio and the share of our tokens found in their name (unaccented)."""
    a = " ".join(sorted(name_tokens(ours)))
    b = " ".join(sorted(name_tokens(theirs)))
    ours_t, theirs_t = name_tokens(ours), name_tokens(theirs)
    overlap = len(ours_t & theirs_t) / len(ours_t) if ours_t else 0.0
    return max(difflib.SequenceMatcher(None, a, b).ratio(), overlap)


def component(place, *types):
    for c in place.get("addressComponents", []):
        if any(t in c.get("types", []) for t in types):
            return c.get("longText")
    return None


def ward_from_address(formatted_address):
    """Post-2025 addresses end with '<ward>, <city>': Google often has the ward only in the text."""
    parts = [p.strip() for p in (formatted_address or "").split(",")]
    return parts[-2] if len(parts) >= 3 else None


def geocode(row):
    city = row["city"] or "Hồ Chí Minh"
    parts = [row["restaurant_name"], row["address"], row["district"], city]
    query = ", ".join(p for p in parts if p)
    body = {"textQuery": query, "languageCode": "vi", "regionCode": "VN", "pageSize": 5}
    if "chi minh" in unaccent_vi(city).lower():
        body["locationBias"] = HCMC_BIAS
    resp = requests.post(
        "https://places.googleapis.com/v1/places:searchText",
        headers={"X-Goog-Api-Key": API_KEY, "X-Goog-FieldMask": FIELD_MASK},
        json=body, timeout=30,
    )
    if resp.status_code != 200:
        return None, f"HTTP {resp.status_code}: {resp.text[:200]}"
    places = resp.json().get("places", [])
    out = {"restaurant_name_normalized": row["restaurant_name_normalized"], "district_norm": row["district_norm"],
           "query_text": query}
    if not places:
        return {**out, "match_status": "no_result"}, None
    status = "not_food"
    for p in places:
        types = set(p.get("types", []))
        if not (types & FOOD_TYPES or any(t.endswith("_restaurant") for t in types)):
            continue
        sim = name_similarity(row["restaurant_name"], (p.get("displayName") or {}).get("text"))
        if sim < MIN_NAME_SIMILARITY:
            status = "name_mismatch"
            continue
        loc = p.get("location") or {}
        return {**out, "match_status": "matched", "place_id": p.get("id"),
                "place_name": (p.get("displayName") or {}).get("text"),
                "formatted_address": p.get("formattedAddress"),
                "ward": (component(p, "sublocality_level_1", "administrative_area_level_3", "sublocality")
                         or ward_from_address(p.get("formattedAddress"))),
                "city": component(p, "administrative_area_level_1"),
                "latitude": loc.get("latitude"), "longitude": loc.get("longitude"),
                "place_types": sorted(types), "name_similarity": round(sim, 3)}, None
    return {**out, "match_status": status}, None


results, errors, api_calls = [], [], 0
for row in todo[:budget]:
    result, error = geocode(row.asDict())
    api_calls += 1
    if error and error.startswith(("HTTP 401", "HTTP 403")):
        raise RuntimeError(f"Places API rejected the key (check secret maps/api_key and the API restrictions): {error}")
    if error and error.startswith("HTTP 429"):
        # Quota used up (e.g. 'SearchTextRequest per day'): stop here, the rest is looked up on a later run
        errors.append(f"{row['restaurant_name']}: {error}")
        print(f"⏸️ Places API quota reached after {api_calls} call(s); {len(todo) - len(results)} key(s) wait for the next run")
        break
    if error:
        errors.append(f"{row['restaurant_name']}: {error}")
    else:
        results.append(result)
print(f"API calls: {api_calls} · cached results: {len(results)} · errors (retried next run): {len(errors)}")
for e in errors[:5]:
    print("  ⚠️", e[:300])
if len(todo) > budget:
    print(f"⏸️ Daily cap: {len(todo) - budget} key(s) wait for tomorrow")

# Record the calls first, so the daily cap holds even if a later step fails
spark.sql(f"""
    INSERT INTO {CALLS_TABLE}
    VALUES (current_timestamp(), DATE'{today}', {api_calls}, {len(results)}, {len(errors)})
""")


# --- Free lookups (2026-11-03): Photon (OpenStreetMap) point + match with a Google place found by auto_search ---
import math
import time

google_done = {(r["restaurant_name_normalized"], r["district_norm"]) for r in results}
free_todo = [r.asDict() for r in todo if (r["restaurant_name_normalized"], r["district_norm"]) not in google_done
             and not any(e.startswith(f"{r['restaurant_name']}:") for e in errors)]


def distance_m(lat1, lng1, lat2, lng2):
    p1, p2 = math.radians(lat1), math.radians(lat2)
    h = (math.sin((p2 - p1) / 2) ** 2
         + math.cos(p1) * math.cos(p2) * math.sin(math.radians(lng2 - lng1) / 2) ** 2)
    return 2 * 6371000 * math.asin(math.sqrt(h))


def is_hcmc(city):
    return "chi minh" in unaccent_vi(city or "Hồ Chí Minh").lower()


def photon(row):
    """Approximate point of the address said in the video, or None (no address with a number / no result)."""
    address = (row["address"] or "").strip()
    if not re.search(r"\d", address):
        return None
    parts = [address] + [x for x in (row["district"], row["city"] or "Hồ Chí Minh")
                         if x and unaccent_vi(x).lower() not in unaccent_vi(address).lower()]
    params = {"q": ", ".join(parts) + ", Việt Nam", "limit": 1}
    if is_hcmc(row["city"]):
        params["bbox"] = HCMC_BBOX
    try:
        resp = requests.get(PHOTON_URL, params=params, headers=PHOTON_HEADERS, timeout=PHOTON_TIMEOUT_SECONDS)
        features = resp.json().get("features", []) if resp.status_code == 200 else []
    except (requests.RequestException, ValueError):
        return None
    if not features or (features[0].get("properties") or {}).get("countrycode", "VN").upper() != "VN":
        return None
    f = features[0]
    lng, lat = f["geometry"]["coordinates"]
    props = f.get("properties") or {}
    label = ", ".join(x for x in (props.get("housenumber"), props.get("street") or props.get("name"),
                                  props.get("district") or props.get("locality"), props.get("city")) if x)
    return {"latitude": lat, "longitude": lng, "label": label or None, "city": props.get("city") or props.get("state")}


places = []
if spark.catalog.tableExists(PLACES_TABLE):
    places = [r.asDict() for r in spark.sql(f"""
        SELECT place_id, place_name, restaurant_name_normalized, formatted_address, ward, city, latitude, longitude,
               place_types FROM {PLACES_TABLE} WHERE place_id IS NOT NULL AND latitude IS NOT NULL""").collect()]


def distinctive_tokens(name):
    """Words of a name that are not a dish / kind of place / number ("20k"): what identifies the restaurant."""
    return {t for t in name_tokens(name) if t not in GENERIC_NAME_WORDS and not re.fullmatch(r"\d+k?", t)}


def specific_match(ours, theirs):
    """Our name has a distinctive word and the Google name has it too (so "Bún riêu cua" never matches)."""
    mine = distinctive_tokens(ours)
    return bool(mine) and bool(mine & name_tokens(theirs))


def match_place(name, name_norm, city, lat, lng):
    """Best Google place (already found, no call) for a mention: similar, specific name, close to the point.

    No point (no address in the video) = no match: chains (Pizza Hut, Kimochi Sushi) have several branches.
    """
    if lat is None:
        return None
    best = None
    for p in places:
        d = distance_m(lat, lng, p["latitude"], p["longitude"])
        if d > PLACES_MATCH_MAX_M:
            continue
        sim = name_similarity(name, p["place_name"])
        if (sim >= PLACES_MIN_SIMILARITY and specific_match(name, p["place_name"])
                and (best is None or (sim, -d) > (best[1], -best[2]))):
            best = (p, sim, d)
    return best


def places_row(key, p, sim, query):
    return {**key, "query_text": query, "match_status": "places_match", "place_id": p["place_id"],
            "place_name": p["place_name"], "formatted_address": p["formatted_address"], "ward": p["ward"],
            "city": p["city"], "latitude": p["latitude"], "longitude": p["longitude"],
            "place_types": p["place_types"], "name_similarity": round(sim, 3)}


photon_calls, free_results = 0, []
photon_start = time.time()
for row in free_todo:
    key = {"restaurant_name_normalized": row["restaurant_name_normalized"], "district_norm": row["district_norm"]}
    query = ", ".join(x for x in (row["restaurant_name"], row["address"], row["district"], row["city"]) if x)
    point = None
    if (photon_calls < PHOTON_MAX_PER_RUN and time.time() - photon_start < PHOTON_BUDGET_SECONDS
            and re.search(r"\d", row["address"] or "")):
        point = photon(row)
        photon_calls += 1
        time.sleep(1.1)   # Photon fair use
    lat, lng = (point["latitude"], point["longitude"]) if point else (None, None)
    best = match_place(row["restaurant_name"], row["restaurant_name_normalized"], row["city"], lat, lng)
    if best:
        free_results.append(places_row(key, best[0], best[1], query))
    elif point:
        free_results.append({**key, "query_text": query, "match_status": "osm_approx",
                             "place_name": row["restaurant_name"], "formatted_address": row["address"],
                             "city": row["city"] or point["city"], "latitude": lat, "longitude": lng})
    # neither: not cached, tried again next run (when Photon budget / new Google places allow)

# places_match rows from the first, looser rule (2026-11-03 run): drop the ones the stricter rule rejects
# (no address in the video, or a generic name); they are looked up again (Photon) on the next run
loose = spark.sql(f"""SELECT restaurant_name_normalized, district_norm, query_text, place_name
                      FROM ({KEYS_SQL}) WHERE match_status = 'places_match'""").collect()
drop = [r for r in loose
        if not re.search(r"\d", r["query_text"] or "")
        or not specific_match((r["query_text"] or "").split(",")[0], r["place_name"])]
if drop:
    spark.createDataFrame([(r["restaurant_name_normalized"], r["district_norm"]) for r in drop],
                          "restaurant_name_normalized STRING, district_norm STRING").createOrReplaceTempView("geocode_drop")
    spark.sql(f"""MERGE INTO {MENTIONS_TABLE} t USING geocode_drop s
                  ON t.restaurant_name_normalized = s.restaurant_name_normalized AND t.district_norm <=> s.district_norm
                 AND t.match_status = 'places_match'
                  WHEN MATCHED THEN UPDATE SET {", ".join(f"t.{c} = NULL" for c in [*GEO_COLS, "geocoded_at"])}""")
print(f"places_match re-checked: {len(loose)} · dropped (looked up again next run): {len(drop)} "
      + str([r["query_text"].split(",")[0] for r in drop]))

# Earlier approximate rows: upgrade to a Google place when auto_search has found it since
upgrades = []
for g in spark.sql(f"""SELECT g.restaurant_name_normalized, g.district_norm, g.query_text, g.place_name, g.city,
                              g.latitude, g.longitude
                       FROM ({KEYS_SQL}) g WHERE g.match_status = 'osm_approx'""").collect():
    best = match_place(g["place_name"], g["restaurant_name_normalized"], g["city"], g["latitude"], g["longitude"])
    if best:
        upgrades.append(places_row({"restaurant_name_normalized": g["restaurant_name_normalized"],
                                    "district_norm": g["district_norm"]}, best[0], best[1], g["query_text"]))
results += free_results + upgrades
print(f"Free lookups: {len(free_todo)} key(s) · Photon calls: {photon_calls} · "
      f"places_match: {sum(r['match_status'] == 'places_match' for r in free_results)} · "
      f"osm_approx: {sum(r['match_status'] == 'osm_approx' for r in free_results)} · "
      f"approx upgraded to a Google place: {len(upgrades)}")


In [ ]:
# --- MERGE the lookups into stg_restaurant_mentions (every row of the key) ---
if results:
    rows = [{**{k: r.get(k) for k in ("restaurant_name_normalized", "district_norm")},
             **{RENAME.get(k, k): v for k, v in r.items() if RENAME.get(k, k) in GEO_COLS}} for r in results]
    rows = [{c: row.get(c) for c in ["restaurant_name_normalized", "district_norm", *GEO_COLS]} for row in rows]
    spark.createDataFrame(rows, schema="restaurant_name_normalized STRING, district_norm STRING, "
                          + ", ".join(f"{c} {t}" for c, t in GEO_COLS.items())).createOrReplaceTempView("geocode_src")
    spark.sql(f"""
        MERGE INTO {MENTIONS_TABLE} AS t
        USING geocode_src AS s
        ON t.restaurant_name_normalized = s.restaurant_name_normalized AND t.district_norm <=> s.district_norm
        WHEN MATCHED AND (t.match_status IS NULL
                          OR t.match_status = 'osm_approx' AND s.match_status = 'places_match') THEN UPDATE SET
            {", ".join(f"t.{c} = s.{c}" for c in GEO_COLS)}, t.geocoded_at = current_timestamp()
    """)
print(f"✅ Merged {len(results)} lookups into {MENTIONS_TABLE}")

# Rows looked up before the ward was parsed from the address text (no API call needed)
spark.sql(f"""
    UPDATE {MENTIONS_TABLE}
    SET geo_ward = trim(element_at(split(formatted_address, ','), -2))
    WHERE geo_ward IS NULL AND match_status = 'matched' AND size(split(formatted_address, ',')) >= 3
""")


In [ ]:
# --- Verify ---
stats = spark.sql(f"""
    SELECT COUNT(*) AS n_keys,
           COUNT_IF(match_status IS NULL) AS not_looked_up,
           COUNT_IF(match_status IN ('matched', 'places_match')) AS google,
           COUNT_IF(match_status = 'osm_approx') AS osm,
           SUM(CASE WHEN n_status > 1 THEN 1 ELSE 0 END) AS inconsistent
    FROM (SELECT restaurant_name_normalized, district_norm, MAX(match_status) AS match_status,
                 COUNT(DISTINCT coalesce(match_status, '-'), coalesce(place_id, '-')) AS n_status
          FROM {MENTIONS_TABLE} GROUP BY restaurant_name_normalized, district_norm)
""").collect()[0]
print(f"📊 {MENTIONS_TABLE}: {stats['n_keys']} restaurant keys · Google place: {stats['google']} · "
      f"OSM point: {stats['osm']} · not located yet: {stats['not_looked_up']}")
print("📊 Same lookup on every row of a key: "
      + ("✅" if stats["inconsistent"] == 0 else f"❌ {stats['inconsistent']} key(s) differ"))
display(spark.sql(f"""SELECT match_status, COUNT(*) AS n_keys FROM (
    SELECT DISTINCT restaurant_name_normalized, district_norm, match_status FROM {MENTIONS_TABLE}) GROUP BY 1 ORDER BY 2 DESC"""))
dbutils.jobs.taskValues.set("api_calls", api_calls)
print("\n--- stg_restaurant_geocode complete ---")
